# Computational Wine Sales Project

For this final assignment you are asked to try to improve on your model for the wine sales data using the Hurdle model. 

#### Deliverables
- A csv file which has the scored records values from test.csv. There will be only two columns in this file: `INDEX`, `P_TARGET`. You will be graded on how your model performs versus my model and those of other students in the class.

#### Scored Data File
Score the data file wine_test.csv. Create a file that has only two variables for each record: 
- `INDEX`
- `P_TARGET`
- Name this file scottkeighley_410_CA_06.csv. The first variable, `INDEX`, will allow you to match your grading to my predicted value.

**Goal**: Improve on the HW04 Poisson model by using a **hurdle model**. This is a two-part structure that separately models (1) whether any cases sell at all, and (2) how many cases sell given that at least one does. This lets the two questions have genuinely different drivers, rather than assuming a single process generates both zeros and positive counts (the assumption a plain Poisson model makes). 

Built manually from two separate fits (`statsmodels`' built-in `HurdleCountModel` requires a single shared variable set for both parts and a Poisson/NegBin zero-model rather than logistic, which doesn't match the assignment's reference R implementation or the variable-selection approach used here) rather than using the combined wrapper class. 

---
### Setup

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.genmod.generalized_linear_model as glm_module
from statsmodels.discrete.truncated_model import TruncatedLFPoisson
from scipy.stats import ttest_ind

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
glm_module.SET_USE_BIC_LLF(True)

In [7]:
train = pd.read_csv('wine_train_logistic.csv', encoding='utf-8-sig')
test = pd.read_csv('wine_test.csv', encoding='utf-8-sig')

print(train.shape, test.shape)
train.head()

(12795, 17) (3335, 16)


,INDEX,TARGET,TARGET_1,FixedAcidity,VolatileAcidity,CitricAcid,ResidualSugar,Chlorides,FreeSulfurDioxide,TotalSulfurDioxide,Density,pH,Sulphates,Alcohol,LabelAppeal,AcidIndex,STARS
0,1,3,1,3.2,1.160,-0.98,54.2,-0.567,NaN,268.0,0.99280,3.33,-0.59,9.9,0,8,2.0
1,2,3,1,4.5,0.160,-0.81,26.1,-0.425,15.0,-327.0,1.02792,3.38,0.70,NaN,-1,7,3.0
2,4,5,1,7.1,2.640,-0.88,14.8,0.037,214.0,142.0,0.99518,3.12,0.48,22.0,-1,8,3.0
3,5,3,1,5.7,0.385,0.04,18.8,-0.425,22.0,115.0,0.99640,2.24,1.83,6.2,-1,6,1.0
4,6,4,1,8.0,0.330,-1.26,9.4,NaN,-167.0,108.0,0.99457,3.12,1.77,13.7,0,9,2.0


---
### Data Preparation (reused from HW03/HW04)

Identical pipeline: sign fix (`abs()`), `STARS_missing` flag, median/mean imputation, standardized quadratic terms. These are properties of the predictors and don't depend on the target or model type. 

In [14]:
sign_cols = ['FixedAcidity', 'VolatileAcidity', 'CitricAcid', 'ResidualSugar',
             'Chlorides', 'FreeSulfurDioxide', 'TotalSulfurDioxide', 'Sulphates']

train_abs = train.copy()
train_abs[sign_cols] = train_abs[sign_cols].abs()

mcar_median_cols = ['Sulphates', 'TotalSulfurDioxide', 'FreeSulfurDioxide', 'Chlorides', 'ResidualSugar']
mcar_mean_cols = ['Alcohol', 'pH']
fill_values = {c: train_abs[c].median() for c in mcar_median_cols}
fill_values.update({c: train_abs[c].mean() for c in mcar_mean_cols})
fill_values['STARS'] = train_abs['STARS'].median()

def prepare_wine_data(df, sign_cols, fill_values):
    """HW03 Section 02 pipeline: sign fix -> STARS_missing flag -> impute (train-derived fill values)."""
    out = df.copy()
    out[sign_cols] = out[sign_cols].abs()
    out['STARS_missing'] = out['STARS'].isna().astype(int)
    for col, val in fill_values.items():
        out[col] = out[col].fillna(val)
    return out

train_prepped = prepare_wine_data(train, sign_cols, fill_values)
test_prepped = prepare_wine_data(test, sign_cols, fill_values)

std_cols = ['Sulphates', 'TotalSulfurDioxide', 'Chlorides', 'Density', 'FixedAcidity', 'FreeSulfurDioxide']
std_means = {col: train_prepped[col].mean() for col in std_cols}
std_stds = {col: train_prepped[col].std() for col in std_cols}

for df in [train_prepped, test_prepped]:
    for col in std_cols:
        df[f'{col}_std'] = (df[col] - std_means[col]) / std_stds[col]
        df[f'{col}_std_sq'] = df[f'{col}_std'] ** 2

print('Remaining missing values, train:')
print(train_prepped.isna().sum()[train_prepped.isna().sum() > 0])
print()
print('Remaining missing values, test:')
print(test_prepped.isna().sum()[test_prepped.isna().sum() > 0])

Remaining missing values, train:
Series([], dtype: int64)

Remaining missing values, test:
TARGET    3335
dtype: int64


--- 
### Variable Selection Strategy

**Hurdle-part (zero vs. positive)**: reuses HW03's BIC-selected logistic variables directly. This is the exact same modeling problem (`TARGET_1`) already rigorously solved in HW03. 

**Count-part (magnitude given a sale)**: re-run stepwise selection, restricted to records where `TARGET > 0`. This subpopulation can have genuinely different variable importance than the full population (e.g. HW03 found `LabelAppeal` predicts case count well sepcifically among wines that sold, while being a flat/weak predictor of sale-vs-no-sale across the whole population). Reusing HW04's full-population Poisson variables for the count-part would risk missing this. 

In [17]:
train_sold = train_prepped[train_prepped['TARGET'] > 0].copy()
print('Full training set:', len(train_prepped))
print('Sold-only subset:', len(train_sold))

Full training set: 12795
Sold-only subset: 10061


In [23]:
full_candidates = [
    'STARS', 'STARS_missing', 'AcidIndex', 'LabelAppeal',
    'VolatileAcidity', 'CitricAcid', 'Alcohol', 'pH', 'ResidualSugar',
    'Sulphates_std', 'Sulphates_std_sq',
    'TotalSulfurDioxide_std', 'TotalSulfurDioxide_std_sq',
    'Chlorides_std', 'Chlorides_std_sq',
    'Density_std', 'Density_std_sq',
    'FixedAcidity_std', 'FixedAcidity_std_sq',
    'FreeSulfurDioxide_std', 'FreeSulfurDioxide_std_sq',
]

def fit_poisson(vars_list, y, df):
    X = sm.add_constant(df[vars_list]) if vars_list else pd.DataFrame({'const': np.ones(len(df))}, index=df.index)
    return sm.GLM(y, X, family=sm.families.Poisson()).fit()

def stepwise_poisson(y, df, candidates, criterion='aic', verbose=True):
    """Bidirectional stepwise selection for Poisson regression, driven by AIC or BIC."""
    included = []
    def score(vars_list):
        m = fit_poisson(vars_list, y, df)
        return m.aic if criterion == 'aic' else m.bic

    current_score = score(included)
    improved = True
    while improved: 
        improved = False
        excluded = [v for v in candidates if v not in included]
        best_add_score, best_add_var = current_score, None
        for v in excluded: 
            trial = included + [v]
            try: 
                s = score(trial)
            except Exception:
                continue
            if s < best_add_score: 
                best_add_score, best_add_var = s, v
        if best_add_var is not None: 
            included.append(best_add_var)
            current_score = best_add_score
            improved = True
            if verbose: 
                print(f'+ add {best_add_var:30s} {criterion.upper()}={current_score:.2f}')

        best_drop_score, best_drop_var = current_score, None
        for v in included:
            trial = [x for x in included if x != v]
            try:
                s = score(trial)
            except Exception:
                continue
            if s < best_drop_score:
                best_drop_score, best_drop_var = s, v
        if best_drop_var is not None:
            included.remove(best_drop_var)
            current_score = best_drop_score
            improved = True
            if verbose:
                print(f'- drop {best_drop_var:30s} {criterion.upper()}={current_score:.2f}')

    return included, current_score

y_sold_count = train_sold['TARGET']

print('=== Count-part stepwise (sold-only subset): AIC ===')
count_aic_vars, count_aic_score = stepwise_poisson(y_sold_count, train_sold, full_candidates, criterion='aic')
print('\nFinal AIC-selected variables:', count_aic_vars)

print('\n=== Count-part stepwise (sold-only subset): BIC ===')
count_bic_vars, count_bic_score = stepwise_poisson(y_sold_count, train_sold, full_candidates, criterion='bic')
print('\nFinal BIC-selected variables:', count_bic_vars)

=== Count-part stepwise (sold-only subset): AIC ===
+ add LabelAppeal                    AIC=34129.04
+ add STARS                          AIC=33891.02
+ add STARS_missing                  AIC=33813.63
+ add Alcohol                        AIC=33791.01
+ add AcidIndex                      AIC=33781.72

Final AIC-selected variables: ['LabelAppeal', 'STARS', 'STARS_missing', 'Alcohol', 'AcidIndex']

=== Count-part stepwise (sold-only subset): BIC ===
+ add LabelAppeal                    BIC=34143.47
+ add STARS                          BIC=33912.67
+ add STARS_missing                  BIC=33842.49
+ add Alcohol                        BIC=33827.09
+ add AcidIndex                      BIC=33825.01

Final BIC-selected variables: ['LabelAppeal', 'STARS', 'STARS_missing', 'Alcohol', 'AcidIndex']


**Result**: AIC and BIC agree -> `LabelAppeal`, `STARS`, `STARS_missing`, `Alcohol`, `AcidIndex`. Notably, `LabelAppeal` is selected first (largest single AIC/BIC improvement) in this sold-only subpopulation, versus being weaker in HW04's full-population model. This is direct/quantifiable confirmation of the suppression-effect finding from HW03. `VolatileAcidity` (present in the hurdle-part) drops out here. `Alcohol` enters instead which is a different best-fit variable set, validating the decision to reselect rather than reuse HW04's list. 

--- 
### Fit Both Parts of the Hurdle Model

**Zero-truncated Poisson, mechanically**: an ordinary Poisson distribution still assigns nonzero probability to `Y=0` for any finite mean. Fitting a plain Poisson to a sold-only subset (where `Y=0` cannot occur by construction) wastes probability mass on an impossible outcome, which systematically biases the fitted mean **downward**. A zero-truncated Poisson redistributes that mass across the achievable outcomes by dividing the Poisson PMF by `1 - P(Y=0)`, correcting this bias. 

#### Part 1: Hurdle (zero) part: logistic regression on `TARGET_1`, full training set

In [26]:
hurdle_vars = ['STARS_missing', 'STARS', 'AcidIndex', 'LabelAppeal', 
               'TotalSulfurDioxide_std', 'pH', 'VolatileAcidity', 'Sulphates_std']

X_hurdle = sm.add_constant(train_prepped[hurdle_vars])
y_binary = train_prepped['TARGET_1']
logit_hurdle = sm.Logit(y_binary, X_hurdle).fit()
print(logit_hurdle.summary())

Optimization terminated successfully.
         Current function value: 0.299981
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:               TARGET_1   No. Observations:                12795
Model:                          Logit   Df Residuals:                    12786
Method:                           MLE   Df Model:                            8
Date:                Thu, 20 Aug 2026   Pseudo R-squ.:                  0.4218
Time:                        20:59:35   Log-Likelihood:                -3838.3
converged:                       True   LL-Null:                       -6637.9
Covariance Type:            nonrobust   LLR p-value:                     0.000
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                      2.6142      0.266      9.841      0.000       2.094      

#### Part 2: Count part: zero-truncated Poisson on `TARGET`, sold-only subset

In [34]:
count_vars = ['LabelAppeal', 'STARS', 'STARS_missing', 'Alcohol', 'AcidIndex']

X_count = sm.add_constant(train_sold[count_vars])
trunc_poisson = TruncatedLFPoisson(y_sold_count, X_count, truncation=0).fit()
print(trunc_poisson.summary())

Optimization terminated successfully.
         Current function value: 1.643543
         Iterations: 14
         Function evaluations: 18
         Gradient evaluations: 18
                    TruncatedLFPoisson Regression Results                     
Dep. Variable:                 TARGET   No. Observations:                10061
Model:             TruncatedLFPoisson   Df Residuals:                    10055
Method:                           MLE   Df Model:                            5
Date:                Thu, 20 Aug 2026   Pseudo R-squ.:                 0.07410
Time:                        21:09:57   Log-Likelihood:                -16536.
converged:                       True   LL-Null:                       -17859.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             1.1510      0.

**Notable finding**: `LabelAppeal`'s coefficient flips sign between the two parts. It is negative (-0.4643) in the hurdle/zero-part, positive (+0.2425) in the count-part. This is the HW03 suppression-effect finding now made explicit and structural, rather than inferred directly. Label appeal appears to work against the initial sale decision (once `STARS` and other quality signals are controlled for) but for higher case counts once wine is already being carried. 

---
### Combining the Two Parts into a Single Prediction

Two candidate combination strategies were tested, on the full training set, against the actual `TARGET` counts: 
- **Expected value blend** (the R/`pscl` default, `predict(..., type="response")`): `P(TARGET>0) * E[TARGET | TARGET>0], rounded.
- **Hard threshold**: if `P(TARGET>0) >= 0.5`, predict `round(E[TARGET | TARGET>0])`; otherwise 0

In [36]:
X_hurdle_full = sm.add_constant(train_prepped[hurdle_vars], has_constant='add')
X_count_full = sm.add_constant(train_prepped[count_vars], has_constant='add')

pi_hat = logit_hurdle.predict(X_hurdle_full)     
mu_trunc = trunc_poisson.predict(X_count_full)   

y_true = train_prepped['TARGET']

# Method A: expected value blend
pred_expected_rounded = np.round(pi_hat * mu_trunc)

# Method B: hard threshold
threshold = 0.5
pred_threshold = np.where(pi_hat >= threshold, np.round(mu_trunc), 0)

mae_expected = np.abs(y_true - pred_expected_rounded).mean()
mae_threshold = np.abs(y_true - pred_threshold).mean()

print('MAE (expected value blend, rounded):', mae_expected)
print('MAE (hard threshold at 0.5):', mae_threshold)

MAE (expected value blend, rounded): 0.9420085971082454
MAE (hard threshold at 0.5): 0.8981633450566627


**Result**: the threshold approach (MAE = 0.898) beats the expected-value blend (MAE = 0.942), despite the blend being the standard formula. Mechanically, the expected-value blend multiplies every prediction by `pi_hat`, which is typically well below 1.0 even for wines the model is fairly confident did sell. This systematically shrinks predictions toward zero across the board. The threshold approach commits fully to the truncated Poisson's (unbiased) count estimate once the hurdle is cleared, with no such discount. The lesson is that the theoretically "proper" formula optimizes for an unbiased probabilistic estimate of the mean, which isn't necessarily what minimizes MAE against integer ground truth. 

**Decision: use the hard threshold (0.5) combination for the final model**. 

--- 
### Baseline Comparison

In [39]:
X_hurdle_test = sm.add_constant(test_prepped[hurdle_vars], has_constant='add')
X_count_test = sm.add_constant(test_prepped[count_vars], has_constant='add')

pi_hat_test = logit_hurdle.predict(X_hurdle_test)
mu_trunc_test = trunc_poisson.predict(X_count_test)

p_target_test = np.where(pi_hat_test >= 0.5, np.round(mu_trunc_test), 0).astype(int)

scored = pd.DataFrame({
    'INDEX': test_prepped['INDEX'], 
    'P_TARGET': p_target_test
})

print(scored.shape)
scored.head(10)

(3335, 2)


,INDEX,P_TARGET
0,3,3
1,9,4
2,10,3
3,18,3
4,21,0
5,30,6
6,31,4
7,37,0
8,39,0
9,47,0


#### Sanity Checks


In [42]:
print('Missing INDEX values:', scored['INDEX'].isna().sum())
print('Missing P_TARGET values:', scored['P_TARGET'].isna().sum())
print('Row count matches wine_test.csv:', len(scored) == len(test_prepped))
print('Any negative P_TARGET values:', (scored['P_TARGET'] < 0).any())
print()
print('P_TARGET distribution:')
print(scored['P_TARGET'].value_counts().sort_index())
print()
print('Mean predicted P_TARGET:', scored['P_TARGET'].mean(),
      ' (training actual mean TARGET:', y_true.mean(), ')')
print('Pct predicted zero:', (scored['P_TARGET'] == 0).mean() * 100, 
      ' (training actual pct zero:', (y_true ==0).mean() * 100, ')')

Missing INDEX values: 0
Missing P_TARGET values: 0
Row count matches wine_test.csv: True
Any negative P_TARGET values: False

P_TARGET distribution:
P_TARGET
0     650
2      83
3    1051
4     925
5     437
6     130
7      54
8       5
Name: count, dtype: int64

Mean predicted P_TARGET: 3.11904047976012  (training actual mean TARGET: 3.0290738569753812 )
Pct predicted zero: 19.490254872563717  (training actual pct zero: 21.36772176631497 )


**Result**: no missing / negative values, row count matches `wine_test.csv`, and both the mean predicted count and the predicted zero-rate closely track the training set's actaul values. This is a reasonable sanity signal the model generalizes without obvious miscalibration

#### Export final scored file

In [45]:
scored.to_csv('scottkeighley_410_CA_06.csv', index=False)
print('Saved scottkeighley_410_CA_06.csv')
scored.head()

Saved scottkeighley_410_CA_06.csv


,INDEX,P_TARGET
0,3,3
1,9,4
2,10,3
3,18,3
4,21,0
